# Day 2 · 05. Self-RAG on LangGraph

## 학습 목표
- LangGraph `StateGraph`와 `TypedDict` 상태로 **자가교정 RAG**를 구현할 수 있다.
- Self-RAG의 문서 관련성 평가 → (관련 있으면) 생성 / (없으면) 쿼리 재작성 후 재검색 루프를 설계할 수 있다.
- 04 Naive RAG의 무한 루프가 **왜** 깨지는지 — `rewrite_query` 노드의 역할을 말할 수 있다.

## 핵심 키워드
LangGraph · StateGraph · TypedDict · Self-RAG · grade_documents · rewrite_query · 자가교정

## 이 노트북의 위치
```
04 (Naive + 관련성 체크, 무한 루프)   →   05 (쿼리 재작성으로 루프 깨기)   →   06 (쿼리 복잡도로 경로 자체를 분기)
```
04의 관련성 체크는 "관련 없음"을 감지하는 데는 성공했지만, 같은 질문으로 돌아갔기에 같은 결과가 반복됐다. 여기서는 `rewrite_query` 한 노드를 끼워 그 고리를 깬다.

In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '..')

from common import get_chat_model, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


## 1. Self-RAG의 핵심 아이디어 — "LLM이 자기 검색 결과를 스스로 심사한다"

04에서 만든 그래프는 `relevance_check` 노드 하나로 "관련성" 을 판단했다. Self-RAG는 여기서 한 걸음 더 간다.

1. **문서 단위로 평가한다** — 전체 context를 통째로 보는 대신, 각 문서를 따로 yes/no로 판단해 살아남은 것만 남긴다.
2. **관련 문서가 0건이면 질문을 고친다** — `rewrite_query` 노드가 LLM에게 "검색이 잘 되도록 다시 써봐" 라고 시킨 뒤 새 질문으로 재검색한다.
3. **재시도 상한을 둔다** — `retries` 카운터로 무한 루프를 방지한다.

이 세 가지가 04의 "같은 질문 → 같은 결과 → 무한 루프" 문제를 풀어주는 차이다.

> 💡 원 논문의 Self-RAG는 reflection token이 찍히도록 fine-tuning된 전용 모델을 쓴다. 여기서는 일반 LLM + 짧은 프롬프트로 **같은 원리를 미니멀하게** 재현한다.

> 📝 **참고**: Self-RAG와 자주 비교되는 변형으로 **CRAG(Corrective RAG)** 가 있다. 평가를 전용 분류기(retrieval evaluator)에 맡기고 품질이 낮을 땐 외부 검색으로 보강한다는 차이가 있지만, airgap 환경에서는 외부 검색이 어려워 본 과정에서는 다루지 않는다.

## 2. 상태 기계 그래프 설계

```
        ┌──────────┐
        │ retrieve │
        └────┬─────┘
             ▼
      ┌──────────────┐      관련 없음     ┌──────────────┐
      │ grade_docs   │──────────────────▶│ rewrite_query│
      └──────┬───────┘                   └──────┬───────┘
             │ 관련 있음                         │
             ▼                                  │
        ┌──────────┐                            │
        │ generate │◀───────────────────────────┘
        └──────────┘             (재시도)
```


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# 공통 인덱스 준비
docs = load_any('../data/corpus_ko.txt')
chunks = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40).split_documents(docs)
vectordb = FAISS.from_documents(chunks, get_embeddings())
retriever = vectordb.as_retriever(search_kwargs={'k': 3})
llm = get_chat_model(temperature=0.0)


## 3. 상태 정의와 노드 구현

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

class RagState(TypedDict):
    question: str             # 현재(재작성 포함) 질문
    original_question: str    # 사용자가 처음 준 질문
    documents: list[Document]
    generation: str
    retries: int              # 재시도 횟수 상한용
    log: list[str]            # 디버그 로그


In [ ]:
# --- 노드 1: retrieve ---
def retrieve(state: RagState) -> RagState:
    q = state['question']
    docs = retriever.invoke(q)
    state['documents'] = docs
    state['log'] = state.get('log', []) + [f'[retrieve] q="{q}" -> {len(docs)}건']
    return state


In [ ]:
# --- 노드 2: grade_documents ---
grade_prompt = ChatPromptTemplate.from_template(
    '다음 문서가 질문에 답하는 데 **관련 있는지** 판단하라.\n'
    '관련 있으면 yes, 없으면 no 한 단어만 출력.\n\n질문: {q}\n문서: {d}'
)
grade_chain = grade_prompt | llm | StrOutputParser()

def grade_documents(state: RagState) -> RagState:
    q = state['question']
    good = []
    for d in state['documents']:
        verdict = grade_chain.invoke({'q': q, 'd': d.page_content}).strip().lower()
        if verdict.startswith('yes'):
            good.append(d)
    state['documents'] = good
    state['log'].append(f'[grade] 관련 문서 {len(good)}건 유지')
    return state


In [ ]:
# --- 노드 3: rewrite_query ---
rewrite_prompt = ChatPromptTemplate.from_template(
    '다음 질문을 더 명확하고 검색이 잘 되도록 다시 써라. 한 줄만 출력:\n{q}'
)
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

def rewrite_query(state: RagState) -> RagState:
    new_q = rewrite_chain.invoke({'q': state['question']}).strip()
    state['question'] = new_q
    state['retries'] = state.get('retries', 0) + 1
    state['log'].append(f'[rewrite] 새 질문: "{new_q}" (retry={state["retries"]})')
    return state


In [ ]:
# --- 노드 4: generate ---
gen_prompt = ChatPromptTemplate.from_template(
    '아래 컨텍스트만 사용해 한국어로 답하라. 근거가 없으면 "근거 없음"이라고 답하라.\n\n'
    '컨텍스트:\n{ctx}\n\n질문: {q}\n답변:'
)
gen_chain = gen_prompt | llm | StrOutputParser()

def generate(state: RagState) -> RagState:
    ctx = '\n---\n'.join(d.page_content for d in state['documents'])
    ans = gen_chain.invoke({'ctx': ctx, 'q': state['original_question']})
    state['generation'] = ans
    state['log'].append(f'[generate] {len(ans)}자 답변 생성')
    return state


In [ ]:
# --- 분기 함수: grade 후 어디로 갈지 결정 ---
def decide_after_grade(state: RagState) -> str:
    if len(state['documents']) == 0 and state.get('retries', 0) < 2:
        return 'rewrite'
    return 'generate'


## 4. StateGraph 조립 및 컴파일

In [ ]:
from langgraph.graph import StateGraph, END

g = StateGraph(RagState)
g.add_node('retrieve', retrieve)
g.add_node('grade', grade_documents)
g.add_node('rewrite', rewrite_query)
g.add_node('generate', generate)

g.set_entry_point('retrieve')
g.add_edge('retrieve', 'grade')
g.add_conditional_edges('grade', decide_after_grade, {
    'rewrite': 'rewrite',
    'generate': 'generate',
})
g.add_edge('rewrite', 'retrieve')   # 다시 검색
g.add_edge('generate', END)

app = g.compile()
print('Self-RAG 그래프 컴파일 완료')


In [ ]:
# 그래프 시각화 (Self-RAG app): Mermaid DSL 텍스트 + PNG 렌더(가능한 환경에서만)
print('--- Mermaid DSL (Self-RAG app) ---')
print(app.get_graph().draw_mermaid())

try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'\n[PNG 렌더 실패 — 텍스트 버전으로 대체] {type(e).__name__}: {e}')

## 5. End-to-End 실행 & 로그 확인

In [ ]:
question = '망분리는 왜 금융권에서 의무화되어 있는가?'
init_state: RagState = {
    'question': question,
    'original_question': question,
    'documents': [],
    'generation': '',
    'retries': 0,
    'log': [],
}
final = app.invoke(init_state)

print('=== 실행 로그 ===')
for line in final['log']:
    print(line)
print('\n=== 최종 답변 ===')
print(final['generation'])


## 6. 다음 노트북 (06 Adaptive RAG) 으로 이어가기

04·05에서 본 구조는 전부 **"검색을 먼저 한 뒤 품질을 사후 평가"** 하는 방식이었다.
- 04: 관련 없으면 다시 검색 (같은 질문이라 실패)
- 05: 관련 없으면 **질문을 바꿔서** 다시 검색 (루프를 깸)

06 Adaptive RAG는 시점을 앞당긴다. **질문을 받자마자 먼저 분류해서**, 애초에 어떤 경로로 답할지를 결정한다.

| 접근 | 결정 시점 | 대표 노드 | 본 과정에서의 위치 |
|---|---|---|---|
| Self-RAG (05) | 검색 **후** 평가 → 필요하면 재검색 | `grade_documents` · `rewrite_query` | **지금 이 노트북** |
| Adaptive RAG (06) | 검색 **전** 경로 결정 | `classify_query` · `route` | 다음 노트북 |

> 💡 실무에서는 둘을 섞어 쓴다. 06의 `classify_query` 가 "단일 검색" 경로를 고른 뒤, 그 경로 안에서 다시 Self-RAG 스타일로 grade·rewrite 를 돌리는 식이다. 원리는 이 노트북에서 본 대로다.

> 📝 **참고 — CRAG로의 확장**: Self-RAG를 발전시킨 변형으로, LLM 대신 경량 분류기로 평가하고 품질이 낮으면 외부/보조 검색을 쓰는 CRAG가 있다. airgap 환경에서는 "외부 웹" 대신 **사내 다른 지식베이스**(Wiki Qdrant · 기업 검색 등)로 fallback하는 패턴이 현실적이다.